# Advanced Model Optimization Pipeline
## Cải thiện chất lượng dự đoán cho House Prices Competition

Notebook này tập trung vào các kỹ thuật tối ưu hóa nâng cao:
- ✨ Advanced Feature Engineering
- 🎯 Hyperparameter Tuning với Optuna
- 🔄 Advanced Ensemble Methods
- 🏗️ Multi-level Stacking
- 📊 Model Diagnostics & Error Analysis

**Mục tiêu**: Cải thiện RMSLE từ ~0.11 xuống < 0.10

## 1. Advanced Feature Engineering
### Tạo các features phức tạp và domain-specific

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.feature_selection import SelectKBest, f_regression, RFE
from sklearn.linear_model import LassoCV
import warnings
warnings.filterwarnings('ignore')

# Load processed data from previous notebook
train_df = pd.read_csv('../data/raw/train.csv')
test_df = pd.read_csv('../data/raw/test.csv')

print(f"✅ Loaded data - Train: {train_df.shape}, Test: {test_df.shape}")

In [ ]:
# Recreate the preprocessing pipeline from global_preprocessing
print("🔄 Applying preprocessing pipeline...")

# Remove outliers
train_df = train_df.drop(train_df[(train_df['GrLivArea'] > 4000) & 
                                   (train_df['SalePrice'] < 300000)].index)

# Save IDs and target
train_ID = train_df['Id']
test_ID = test_df['Id']
y_train = np.log1p(train_df['SalePrice'].values)

# Drop Id and SalePrice
train_df.drop(['Id', 'SalePrice'], axis=1, inplace=True)
test_df.drop('Id', axis=1, inplace=True)

# Combine datasets
ntrain = train_df.shape[0]
ntest = test_df.shape[0]
all_data = pd.concat([train_df, test_df]).reset_index(drop=True)

print(f"✅ Combined data shape: {all_data.shape}")
print(f"✅ Target (log transformed) range: [{y_train.min():.2f}, {y_train.max():.2f}]")

In [ ]:
# Handle missing values (same as global_preprocessing)
print("\n🔧 Handling missing values...")

# Categorical features that mean 'None' when missing
none_cols = ['PoolQC', 'MiscFeature', 'Alley', 'Fence', 'FireplaceQu',
             'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond',
             'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
             'MasVnrType']

for col in none_cols:
    if col in all_data.columns:
        all_data[col] = all_data[col].fillna('None')

# Numerical features that mean 0 when missing
zero_cols = ['GarageYrBlt', 'GarageArea', 'GarageCars',
             'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF',
             'BsmtFullBath', 'BsmtHalfBath', 'MasVnrArea']

for col in zero_cols:
    if col in all_data.columns:
        all_data[col] = all_data[col].fillna(0)

# LotFrontage - fill by neighborhood median
all_data['LotFrontage'] = all_data.groupby('Neighborhood')['LotFrontage'].transform(
    lambda x: x.fillna(x.median()))

# Mode imputation for remaining categorical
mode_cols = ['MSZoning', 'Functional', 'Electrical', 'KitchenQual',
             'Exterior1st', 'Exterior2nd', 'SaleType']

for col in mode_cols:
    if col in all_data.columns:
        all_data[col] = all_data[col].fillna(all_data[col].mode()[0])

# Drop Utilities (low variance)
all_data = all_data.drop(['Utilities'], axis=1, errors='ignore')

# Convert MSSubClass to string
all_data['MSSubClass'] = all_data['MSSubClass'].apply(str)

print(f"✅ Missing values remaining: {all_data.isnull().sum().sum()}")

### 1.1 Polynomial & Interaction Features
Tạo các tương tác giữa features quan trọng

In [ ]:
print("\n🎨 Creating Advanced Features...")

# === BASIC AGGREGATIONS (from previous notebook) ===
all_data['TotalSF'] = all_data['TotalBsmtSF'] + all_data['1stFlrSF'] + all_data['2ndFlrSF']
all_data['TotalPorch'] = (all_data['OpenPorchSF'] + all_data['EnclosedPorch'] +
                           all_data['3SsnPorch'] + all_data['ScreenPorch'])
all_data['TotalArea'] = (all_data['TotalBsmtSF'] + all_data['1stFlrSF'] +
                          all_data['2ndFlrSF'] + all_data['GarageArea'])

all_data['TotalBath'] = (all_data['FullBath'] + 0.5 * all_data['HalfBath'] +
                          all_data['BsmtFullBath'] + 0.5 * all_data['BsmtHalfBath'])

# === TIME FEATURES ===
all_data['HouseAge'] = all_data['YrSold'] - all_data['YearBuilt']
all_data['YearsSinceRemodel'] = all_data['YrSold'] - all_data['YearRemodAdd']
all_data['TotalYears'] = all_data['YearBuilt'] + all_data['YearRemodAdd']

# === BOOLEAN FEATURES ===
all_data['HasPool'] = (all_data['PoolArea'] > 0).astype(int)
all_data['Has2ndFloor'] = (all_data['2ndFlrSF'] > 0).astype(int)
all_data['HasGarage'] = (all_data['GarageArea'] > 0).astype(int)
all_data['HasBsmt'] = (all_data['TotalBsmtSF'] > 0).astype(int)
all_data['HasFireplace'] = (all_data['Fireplaces'] > 0).astype(int)

# === NEW: DOMAIN-SPECIFIC FEATURES ===
# Price per square foot indicators
all_data['PricePerSF'] = all_data['TotalSF'] / (all_data['OverallQual'] + 1)
all_data['LivingAreaPerRoom'] = all_data['GrLivArea'] / (all_data['TotRmsAbvGrd'] + 1)

# Quality interactions
all_data['Quality*Condition'] = all_data['OverallQual'] * all_data['OverallCond'].astype(float)
all_data['Quality*Area'] = all_data['OverallQual'] * all_data['GrLivArea']
all_data['Quality*TotalSF'] = all_data['OverallQual'] * all_data['TotalSF']

# Area ratios
all_data['LivingAreaRatio'] = all_data['GrLivArea'] / (all_data['TotalSF'] + 1)
all_data['GarageRatio'] = all_data['GarageArea'] / (all_data['TotalArea'] + 1)
all_data['BasementRatio'] = all_data['TotalBsmtSF'] / (all_data['TotalArea'] + 1)
all_data['PorchRatio'] = all_data['TotalPorch'] / (all_data['TotalArea'] + 1)

# Bedroom/Room ratios
all_data['BedPerRoom'] = all_data['BedroomAbvGr'] / (all_data['TotRmsAbvGrd'] + 1)
all_data['BathPerBed'] = all_data['TotalBath'] / (all_data['BedroomAbvGr'] + 1)

# === NEW: POLYNOMIAL FEATURES FOR KEY VARIABLES ===
# Square and cube of important features
all_data['GrLivArea_Squared'] = all_data['GrLivArea'] ** 2
all_data['TotalSF_Squared'] = all_data['TotalSF'] ** 2
all_data['OverallQual_Squared'] = all_data['OverallQual'] ** 2

# Log transforms for skewed area features
all_data['Log_GrLivArea'] = np.log1p(all_data['GrLivArea'])
all_data['Log_LotArea'] = np.log1p(all_data['LotArea'])
all_data['Log_TotalSF'] = np.log1p(all_data['TotalSF'])

# Neighborhood quality interaction
all_data['Neighborhood_Quality'] = all_data.groupby('Neighborhood')['OverallQual'].transform('mean')

print(f"✅ Created {all_data.shape[1] - 79} new features")
print(f"📊 Total features: {all_data.shape[1]}")

### 1.2 Label Encoding & One-Hot Encoding

In [ ]:
from sklearn.preprocessing import LabelEncoder

print("\n🏷️ Encoding categorical features...")

# Ordinal features (có thứ tự)
ordinal_cols = ['FireplaceQu', 'BsmtQual', 'BsmtCond', 'GarageQual', 'GarageCond',
                'ExterQual', 'ExterCond', 'HeatingQC', 'PoolQC', 'KitchenQual',
                'BsmtFinType1', 'BsmtFinType2', 'Functional', 'Fence', 'BsmtExposure',
                'GarageFinish', 'LandSlope', 'LotShape', 'PavedDrive', 'Street',
                'Alley', 'CentralAir', 'MSSubClass', 'OverallCond',
                'YrSold', 'MoSold']

for col in ordinal_cols:
    if col in all_data.columns:
        lbl = LabelEncoder()
        all_data[col] = lbl.fit_transform(all_data[col].astype(str))

print(f"   ✅ Label-encoded {len(ordinal_cols)} ordinal features")

# One-Hot Encoding cho categorical features còn lại
all_data = pd.get_dummies(all_data)
print(f"   ✅ One-hot encoded nominal features")
print(f"📐 Final shape after encoding: {all_data.shape}")

### 1.3 Skewness Correction with Box-Cox

In [ ]:
from scipy.stats import skew
from scipy.special import boxcox1p

print("\n📊 Handling skewed features...")

# Get numeric features
numeric_feats = all_data.select_dtypes(exclude=["object"]).columns

# Calculate skewness
skewed_feats = all_data[numeric_feats].apply(lambda x: skew(x.dropna()))
skewness_df = pd.DataFrame({'Skew': skewed_feats}).sort_values(by='Skew', ascending=False)

# Select features with |skew| > 0.5
threshold = 0.5
skewed_features = skewness_df[abs(skewness_df['Skew']) > threshold].index

print(f"Found {len(skewed_features)} skewed features (|skew| > {threshold})")

# Apply Box-Cox transformation
lam = 0.15
for feat in skewed_features:
    if (all_data[feat] <= 0).any():
        all_data[feat] = np.log1p(all_data[feat] - all_data[feat].min() + 1)
    else:
        all_data[feat] = boxcox1p(all_data[feat], lam)

print(f"✅ Applied transformations to skewed features")

### 1.4 Feature Selection with Multiple Methods

In [ ]:
# Split back to train and test
train = all_data[:ntrain]
test = all_data[ntrain:]

print(f"\n🎯 Feature Selection Analysis...")
print(f"Current number of features: {train.shape[1]}")

# === METHOD 1: SelectKBest with f_regression ===
selector_kbest = SelectKBest(score_func=f_regression, k='all')
selector_kbest.fit(train, y_train)

# Get feature scores
feature_scores = pd.DataFrame({
    'Feature': train.columns,
    'Score': selector_kbest.scores_
}).sort_values('Score', ascending=False)

print(f"\n📊 Top 20 features by F-statistic:")
print(feature_scores.head(20))

# === METHOD 2: Lasso Feature Selection ===
from sklearn.linear_model import Lasso

lasso_selector = Lasso(alpha=0.001, random_state=42)
lasso_selector.fit(train, y_train)

lasso_importance = pd.DataFrame({
    'Feature': train.columns,
    'Importance': np.abs(lasso_selector.coef_)
}).sort_values('Importance', ascending=False)

print(f"\n🎯 Top 20 features by Lasso coefficients:")
print(lasso_importance.head(20))

# === Combine both methods ===
# Features that appear in top 100 of both methods
top_kbest = set(feature_scores.head(100)['Feature'])
top_lasso = set(lasso_importance[lasso_importance['Importance'] > 0]['Feature'])
selected_features = list(top_kbest.union(top_lasso))

print(f"\n✅ Selected {len(selected_features)} important features")
print(f"   • Top KBest: {len(top_kbest)} features")
print(f"   • Lasso non-zero: {len(top_lasso)} features")
print(f"   • Union: {len(selected_features)} features")

# Save for later use
train_selected = train[selected_features]
test_selected = test[selected_features]

print(f"\n📐 Selected data shape: Train={train_selected.shape}, Test={test_selected.shape}")

## 2. Hyperparameter Tuning with Optuna
### Tối ưu hóa tự động với Bayesian Optimization

In [ ]:
# Install Optuna
%pip install optuna -q

import optuna
from sklearn.model_selection import cross_val_score, KFold
from sklearn.metrics import mean_squared_error
import xgboost as xgb
import lightgbm as lgb

print("✅ Installed Optuna for hyperparameter optimization")

### 2.1 Optimize XGBoost

In [ ]:
def objective_xgb(trial):
    """Objective function for XGBoost optimization"""
    
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 1000, 5000),
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 7),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-5, 1.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-5, 10.0, log=True),
        'random_state': 42,
        'n_jobs': -1
    }
    
    model = xgb.XGBRegressor(**params)
    
    # 5-fold cross-validation
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    scores = -cross_val_score(model, train, y_train, 
                               scoring='neg_mean_squared_error',
                               cv=kf, n_jobs=-1)
    
    rmse = np.sqrt(scores.mean())
    return rmse

print("🚀 Starting XGBoost optimization...")
print("This may take 10-30 minutes depending on n_trials...")

# Create study
study_xgb = optuna.create_study(direction='minimize', study_name='xgboost_optimization')

# Optimize (use n_trials=50 for quick test, 200+ for best results)
study_xgb.optimize(objective_xgb, n_trials=50, show_progress_bar=True)

print(f"\n✅ Best XGBoost RMSE: {study_xgb.best_value:.4f}")
print(f"📊 Best parameters:")
for key, value in study_xgb.best_params.items():
    print(f"   • {key}: {value}")

# Save best params
best_xgb_params = study_xgb.best_params

### 2.2 Optimize LightGBM

In [ ]:
def objective_lgb(trial):
    """Objective function for LightGBM optimization"""
    
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 1000, 6000),
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 20, 100),
        'max_depth': trial.suggest_int('max_depth', 3, 9),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 50),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.5, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.5, 1.0),
        'bagging_freq': trial.suggest_int('bagging_freq', 1, 10),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-5, 1.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-5, 10.0, log=True),
        'random_state': 42,
        'n_jobs': -1,
        'verbose': -1
    }
    
    model = lgb.LGBMRegressor(**params)
    
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    scores = -cross_val_score(model, train, y_train,
                               scoring='neg_mean_squared_error',
                               cv=kf, n_jobs=-1)
    
    rmse = np.sqrt(scores.mean())
    return rmse

print("🚀 Starting LightGBM optimization...")

study_lgb = optuna.create_study(direction='minimize', study_name='lightgbm_optimization')
study_lgb.optimize(objective_lgb, n_trials=50, show_progress_bar=True)

print(f"\n✅ Best LightGBM RMSE: {study_lgb.best_value:.4f}")
print(f"📊 Best parameters:")
for key, value in study_lgb.best_params.items():
    print(f"   • {key}: {value}")

best_lgb_params = study_lgb.best_params

### 2.3 Optimize CatBoost

In [ ]:
%pip install catboost -q

from catboost import CatBoostRegressor

def objective_catboost(trial):
    """Objective function for CatBoost optimization"""
    
    params = {
        'iterations': trial.suggest_int('iterations', 1000, 5000),
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
        'depth': trial.suggest_int('depth', 4, 10),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-3, 10.0, log=True),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 1.0),
        'random_strength': trial.suggest_float('random_strength', 0.0, 10.0),
        'border_count': trial.suggest_int('border_count', 32, 255),
        'random_state': 42,
        'verbose': 0
    }
    
    model = CatBoostRegressor(**params)
    
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    scores = -cross_val_score(model, train, y_train,
                               scoring='neg_mean_squared_error',
                               cv=kf, n_jobs=1)  # CatBoost handles parallelization internally
    
    rmse = np.sqrt(scores.mean())
    return rmse

print("🚀 Starting CatBoost optimization...")

study_catboost = optuna.create_study(direction='minimize', study_name='catboost_optimization')
study_catboost.optimize(objective_catboost, n_trials=30, show_progress_bar=True)

print(f"\n✅ Best CatBoost RMSE: {study_catboost.best_value:.4f}")
print(f"📊 Best parameters:")
for key, value in study_catboost.best_params.items():
    print(f"   • {key}: {value}")

best_catboost_params = study_catboost.best_params

### 2.4 Visualization of Optimization Results

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# XGBoost optimization history
optuna.visualization.matplotlib.plot_optimization_history(study_xgb, ax=axes[0])
axes[0].set_title('XGBoost Optimization History')

# LightGBM optimization history
optuna.visualization.matplotlib.plot_optimization_history(study_lgb, ax=axes[1])
axes[1].set_title('LightGBM Optimization History')

# CatBoost optimization history
optuna.visualization.matplotlib.plot_optimization_history(study_catboost, ax=axes[2])
axes[2].set_title('CatBoost Optimization History')

plt.tight_layout()
plt.savefig('../notebooks/optimization_history.png', dpi=300, bbox_inches='tight')
plt.show()

print("💾 Saved optimization history plot")

## 3. Advanced Ensemble Methods
### Tạo ensemble với weighted averaging được tối ưu hóa

In [ ]:
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.kernel_ridge import KernelRidge

print("🎯 Building diverse model pool with optimized hyperparameters...")

# === BASE MODELS WITH OPTIMIZED PARAMETERS ===

# 1. XGBoost (optimized)
model_xgb_opt = xgb.XGBRegressor(**best_xgb_params)

# 2. LightGBM (optimized)
model_lgb_opt = lgb.LGBMRegressor(**best_lgb_params)

# 3. CatBoost (optimized)
model_catboost_opt = CatBoostRegressor(**best_catboost_params)

# 4. Extra Trees
model_et = ExtraTreesRegressor(
    n_estimators=200,
    max_depth=15,
    min_samples_split=10,
    min_samples_leaf=4,
    random_state=42,
    n_jobs=-1
)

# 5. Ridge Regression
model_ridge = Ridge(alpha=10.0, random_state=42)

# 6. ElasticNet
model_enet = ElasticNet(alpha=0.0004, l1_ratio=0.9, random_state=42)

# 7. Kernel Ridge
model_krr = KernelRidge(alpha=0.6, kernel='polynomial', degree=2, coef0=2.5)

print("✅ Created 7 diverse models for ensemble")

### 3.1 Train All Models and Get OOF Predictions

In [ ]:
from sklearn.model_selection import KFold

def get_oof_predictions(model, X, y, X_test, n_folds=5, model_name="Model"):
    """
    Get out-of-fold predictions for stacking
    Returns: oof_train (for training meta-model), oof_test (averaged predictions on test)
    """
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)
    
    oof_train = np.zeros(len(X))
    oof_test = np.zeros(len(X_test))
    oof_test_folds = np.zeros((len(X_test), n_folds))
    
    rmse_scores = []
    
    print(f"\n🔄 Training {model_name}...")
    
    for fold, (train_idx, val_idx) in enumerate(kf.split(X), 1):
        X_train_fold = X.iloc[train_idx] if isinstance(X, pd.DataFrame) else X[train_idx]
        y_train_fold = y[train_idx]
        X_val_fold = X.iloc[val_idx] if isinstance(X, pd.DataFrame) else X[val_idx]
        y_val_fold = y[val_idx]
        
        # Train model
        model.fit(X_train_fold, y_train_fold)
        
        # Predict on validation
        oof_train[val_idx] = model.predict(X_val_fold)
        
        # Predict on test
        oof_test_folds[:, fold-1] = model.predict(X_test)
        
        # Calculate RMSE for this fold
        fold_rmse = np.sqrt(mean_squared_error(y_val_fold, oof_train[val_idx]))
        rmse_scores.append(fold_rmse)
        
        print(f"   Fold {fold}: RMSE = {fold_rmse:.4f}")
    
    # Average test predictions across folds
    oof_test = oof_test_folds.mean(axis=1)
    
    # Overall CV score
    cv_rmse = np.sqrt(mean_squared_error(y, oof_train))
    print(f"   ✅ {model_name} CV RMSE: {cv_rmse:.4f} (±{np.std(rmse_scores):.4f})")
    
    return oof_train, oof_test, cv_rmse

# Dictionary to store all predictions
oof_train_preds = {}
oof_test_preds = {}
cv_scores = {}

# Train all models
models = {
    'XGBoost': model_xgb_opt,
    'LightGBM': model_lgb_opt,
    'CatBoost': model_catboost_opt,
    'ExtraTrees': model_et,
    'Ridge': model_ridge,
    'ElasticNet': model_enet,
    'KernelRidge': model_krr
}

print("="*70)
print("TRAINING ALL BASE MODELS")
print("="*70)

for name, model in models.items():
    oof_train, oof_test, cv_score = get_oof_predictions(
        model, train, y_train, test, n_folds=5, model_name=name
    )
    oof_train_preds[name] = oof_train
    oof_test_preds[name] = oof_test
    cv_scores[name] = cv_score

print("\n" + "="*70)
print("MODEL PERFORMANCE SUMMARY")
print("="*70)
for name, score in sorted(cv_scores.items(), key=lambda x: x[1]):
    print(f"{name:20s}: {score:.4f}")

### 3.2 Optimize Ensemble Weights

In [ ]:
from scipy.optimize import minimize

def ensemble_rmse(weights, predictions, target):
    """Calculate RMSE for weighted ensemble"""
    ensemble_pred = np.zeros(len(target))
    for i, pred in enumerate(predictions):
        ensemble_pred += weights[i] * pred
    return np.sqrt(mean_squared_error(target, ensemble_pred))

def optimize_weights(predictions_list, target):
    """
    Optimize ensemble weights using scipy.optimize.minimize
    predictions_list: list of prediction arrays
    target: actual target values
    """
    n_models = len(predictions_list)
    
    # Initial weights (equal)
    initial_weights = [1.0 / n_models] * n_models
    
    # Constraints: weights sum to 1
    constraints = {'type': 'eq', 'fun': lambda w: np.sum(w) - 1}
    
    # Bounds: each weight between 0 and 1
    bounds = [(0.0, 1.0) for _ in range(n_models)]
    
    # Optimize
    result = minimize(
        ensemble_rmse,
        initial_weights,
        args=(predictions_list, target),
        method='SLSQP',
        bounds=bounds,
        constraints=constraints
    )
    
    return result.x

# Prepare predictions for optimization
model_names = list(oof_train_preds.keys())
predictions_list = [oof_train_preds[name] for name in model_names]

print("🎯 Optimizing ensemble weights...")
print("="*70)

optimal_weights = optimize_weights(predictions_list, y_train)

# Calculate optimized ensemble predictions
optimized_train_pred = np.zeros(len(y_train))
optimized_test_pred = np.zeros(len(test))

for i, name in enumerate(model_names):
    optimized_train_pred += optimal_weights[i] * oof_train_preds[name]
    optimized_test_pred += optimal_weights[i] * oof_test_preds[name]

optimized_rmse = np.sqrt(mean_squared_error(y_train, optimized_train_pred))

print(f"\n📊 OPTIMAL WEIGHTS:")
for name, weight in zip(model_names, optimal_weights):
    print(f"   {name:20s}: {weight:.4f}")

print(f"\n✅ Optimized Ensemble RMSE: {optimized_rmse:.4f}")

# Compare with simple average
simple_avg_pred = np.mean(predictions_list, axis=0)
simple_avg_rmse = np.sqrt(mean_squared_error(y_train, simple_avg_pred))
print(f"📊 Simple Average RMSE: {simple_avg_rmse:.4f}")
print(f"🎯 Improvement: {simple_avg_rmse - optimized_rmse:.4f}")

## 4. Model Stacking with Meta-Features
### Multi-level stacking architecture

In [ ]:
from sklearn.base import BaseEstimator, RegressorMixin, TransformerMixin, clone

class AdvancedStackingRegressor(BaseEstimator, RegressorMixin, TransformerMixin):
    """
    Advanced stacking with multiple meta-models and meta-features
    """
    def __init__(self, base_models, meta_models, n_folds=5, use_features=True):
        """
        base_models: list of (name, model) tuples
        meta_models: list of (name, model) tuples for ensemble of meta-learners
        n_folds: number of CV folds
        use_features: whether to include original features in meta-model
        """
        self.base_models = base_models
        self.meta_models = meta_models
        self.n_folds = n_folds
        self.use_features = use_features
        
    def fit(self, X, y):
        """Fit base and meta models"""
        self.base_models_ = [list() for _ in self.base_models]
        self.meta_models_ = [clone(model) for _, model in self.meta_models]
        
        kf = KFold(n_splits=self.n_folds, shuffle=True, random_state=42)
        
        # Create meta-features from base models
        out_of_fold_predictions = np.zeros((X.shape[0], len(self.base_models)))
        
        print(f"\n🏗️ Training {len(self.base_models)} base models...")
        
        for i, (name, model) in enumerate(self.base_models):
            print(f"   Training base model {i+1}/{len(self.base_models)}: {name}")
            
            for fold_idx, (train_idx, val_idx) in enumerate(kf.split(X, y)):
                instance = clone(model)
                self.base_models_[i].append(instance)
                
                X_train = X.iloc[train_idx] if isinstance(X, pd.DataFrame) else X[train_idx]
                y_train_fold = y[train_idx]
                X_val = X.iloc[val_idx] if isinstance(X, pd.DataFrame) else X[val_idx]
                
                instance.fit(X_train, y_train_fold)
                y_pred = instance.predict(X_val)
                out_of_fold_predictions[val_idx, i] = y_pred
        
        # Create additional meta-features
        meta_features = self._create_meta_features(X, out_of_fold_predictions)
        
        # Train each meta-model
        print(f"\n🎯 Training {len(self.meta_models)} meta-models...")
        self.meta_predictions_ = []
        
        for i, (meta_name, meta_model) in enumerate(self.meta_models):
            print(f"   Training meta-model {i+1}/{len(self.meta_models)}: {meta_name}")
            self.meta_models_[i].fit(meta_features, y)
            
        return self
    
    def _create_meta_features(self, X, base_predictions):
        """Create enhanced meta-features"""
        meta_features = [base_predictions]
        
        # Add statistics of base predictions
        meta_features.append(base_predictions.mean(axis=1).reshape(-1, 1))
        meta_features.append(base_predictions.std(axis=1).reshape(-1, 1))
        meta_features.append(base_predictions.max(axis=1).reshape(-1, 1))
        meta_features.append(base_predictions.min(axis=1).reshape(-1, 1))
        
        # Include original features if specified
        if self.use_features:
            if isinstance(X, pd.DataFrame):
                # Select top features
                important_features = X.iloc[:, :20]  # Top 20 features
                meta_features.append(important_features.values)
            else:
                meta_features.append(X[:, :20])
        
        return np.column_stack(meta_features)
    
    def predict(self, X):
        """Make predictions using stacked models"""
        # Get base model predictions
        base_predictions = np.column_stack([
            np.column_stack([model.predict(X) for model in models]).mean(axis=1)
            for models in self.base_models_
        ])
        
        # Create meta-features
        meta_features = self._create_meta_features(X, base_predictions)
        
        # Get predictions from all meta-models and average
        meta_predictions = np.column_stack([
            meta_model.predict(meta_features)
            for meta_model in self.meta_models_
        ])
        
        return meta_predictions.mean(axis=1)

# Define base and meta models
base_models_for_stacking = [
    ('XGBoost', model_xgb_opt),
    ('LightGBM', model_lgb_opt),
    ('CatBoost', model_catboost_opt),
    ('ExtraTrees', model_et)
]

meta_models = [
    ('Ridge', Ridge(alpha=5.0)),
    ('XGB_Meta', xgb.XGBRegressor(n_estimators=100, learning_rate=0.05, max_depth=3)),
    ('LGB_Meta', lgb.LGBMRegressor(n_estimators=100, learning_rate=0.05, num_leaves=31))
]

print("🏗️ Building Advanced Stacking Model...")
stacker = AdvancedStackingRegressor(
    base_models=base_models_for_stacking,
    meta_models=meta_models,
    n_folds=5,
    use_features=True
)

# Train stacking model
stacker.fit(train, y_train)

# Get predictions
stacking_train_pred = stacker.predict(train)
stacking_test_pred = stacker.predict(test)

stacking_rmse = np.sqrt(mean_squared_error(y_train, stacking_train_pred))
print(f"\n✅ Advanced Stacking RMSE: {stacking_rmse:.4f}")

## 5. Cross-Validation Strategy Optimization
### Implement different CV strategies

In [ ]:
from sklearn.model_selection import RepeatedKFold, StratifiedKFold

# Create stratified bins based on price ranges
def create_price_bins(y, n_bins=5):
    """Create stratified bins for cross-validation"""
    bins = pd.qcut(y, q=n_bins, labels=False, duplicates='drop')
    return bins

# Get price bins
price_bins = create_price_bins(y_train, n_bins=5)

print("📊 Comparing Different CV Strategies...")
print("="*70)

# Test with a single model (XGBoost optimized)
test_model = xgb.XGBRegressor(**best_xgb_params)

cv_strategies = {
    'Standard KFold (5)': KFold(n_splits=5, shuffle=True, random_state=42),
    'Standard KFold (10)': KFold(n_splits=10, shuffle=True, random_state=42),
    'Repeated KFold (5x2)': RepeatedKFold(n_splits=5, n_repeats=2, random_state=42),
    'Stratified KFold (5)': StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
}

cv_results = {}

for strategy_name, cv_splitter in cv_strategies.items():
    print(f"\n🔄 Testing: {strategy_name}")
    
    if 'Stratified' in strategy_name:
        scores = cross_val_score(test_model, train, y_train,
                                 scoring='neg_mean_squared_error',
                                 cv=cv_splitter, n_jobs=-1,
                                 fit_params={}, error_score='raise')
        # Use price_bins for stratification
        scores_list = []
        for train_idx, val_idx in cv_splitter.split(train, price_bins):
            X_train_cv = train.iloc[train_idx]
            y_train_cv = y_train[train_idx]
            X_val_cv = train.iloc[val_idx]
            y_val_cv = y_train[val_idx]
            
            test_model.fit(X_train_cv, y_train_cv)
            pred = test_model.predict(X_val_cv)
            score = mean_squared_error(y_val_cv, pred)
            scores_list.append(-score)
        scores = np.array(scores_list)
    else:
        scores = cross_val_score(test_model, train, y_train,
                                 scoring='neg_mean_squared_error',
                                 cv=cv_splitter, n_jobs=-1)
    
    rmse_scores = np.sqrt(-scores)
    cv_results[strategy_name] = {
        'mean': rmse_scores.mean(),
        'std': rmse_scores.std(),
        'scores': rmse_scores
    }
    
    print(f"   RMSE: {rmse_scores.mean():.4f} (±{rmse_scores.std():.4f})")

# Plot comparison
plt.figure(figsize=(12, 6))
strategy_names = list(cv_results.keys())
means = [cv_results[name]['mean'] for name in strategy_names]
stds = [cv_results[name]['std'] for name in strategy_names]

plt.bar(range(len(strategy_names)), means, yerr=stds, capsize=5, alpha=0.7, color='skyblue')
plt.xlabel('CV Strategy')
plt.ylabel('RMSE')
plt.title('Comparison of Cross-Validation Strategies')
plt.xticks(range(len(strategy_names)), strategy_names, rotation=45, ha='right')
plt.tight_layout()
plt.savefig('../notebooks/cv_strategy_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n💾 Saved CV strategy comparison plot")

## 6. Prediction Optimization and Blending
### Optimize final ensemble weights